# 0.Import Libraries

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import re

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

C:\Users\DELL\AppData\Local\Temp\ipykernel_5248\3068766076.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader
c:\Users\DELL\OneDrive\Desktop\rag-assistant-project\notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1.Document Loader

In [2]:
# ============================================================
# 2. File paths
# ============================================================

PDF_PATH = r"C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\raw\alsyra_alnubawia.pdf"

TXT_PATH = r"C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\raw\raw_text.txt"

In [3]:
# ============================================================
# 3. Load PDF
# ============================================================

pdf_loader = PyPDFLoader(PDF_PATH)
pdf_documents = pdf_loader.load()

print(f"PDF pages: {len(pdf_documents)}")

print("\nFirst PDF page:")
print(pdf_documents[0].page_content[:500])

print("\nPDF metadata:")
print(pdf_documents[0].metadata)

PDF pages: 257

First PDF page:


PDF metadata:
{'producer': 'الإصدار التجريبي لـ Microsoft® Word 2010', 'creator': 'الإصدار التجريبي لـ Microsoft® Word 2010', 'creationdate': '2017-08-03T16:43:04+02:00', 'author': 'موسوعة محمد رسول الله صلى الله عليه وسلم', 'moddate': '2017-08-05T13:09:31+02:00', 'subject': 'المواقع الشخصية  >  مواقع المشايخ والعلماء  >  موسوعة محمد رسول الله صلى الله عليه وسلم  >  مختصر السيرة النبوية لابن هشام \tحذف التصنيف\r\nملفات خاصة  >  محمد صلى الله عليه وسلم  >  كتب', 'title': 'مختصر السيرة النبوية لابن هشام (PDF) أ. د. أحمد المزيد', 'source': 'C:\\Users\\DELL\\OneDrive\\Desktop\\rag-assistant-project\\data\\raw\\alsyra_alnubawia.pdf', 'total_pages': 257, 'page': 0, 'page_label': '1'}


In [4]:
# ============================================================
# 4. Load TXT
# ============================================================

txt_loader = TextLoader(
    TXT_PATH,
    encoding="utf-8"
)

txt_documents = txt_loader.load()

print(f"\nTXT documents: {len(txt_documents)}")

print("\nFirst TXT content:")
print(txt_documents[0].page_content[:500])

print("\nTXT metadata:")
print(txt_documents[0].metadata)


TXT documents: 1

First TXT content:
﻿الرحيق المختوم


________________
الحكـم والإمـارة فـي العـرب 
الملك باليمن‏‏ 
الملك بالحيرة 
الملك بالشام 
الإمارة بالحجاز 
الحكم في سائر العرب 
الحالة السياسية 
________________


كان حكام جزيرة العرب عند ظهور دعوة النبي صلى الله عليه وسلم على قسمين‏:‏ 
1ـ ملوك مُتَوَّجُون ـ إلا أنهم في الحقيقة كانوا غير مستقلين‏.‏ 
2ـ رؤسـاء القبائـل والعشائر ـ وكـان لهم مـن الحكم والامتـياز مـا كـان للملـوك المتوجين، ومعظم هـؤلاء كانـوا على تمـام الاستقـلال، وربمـا كانت لبعضـهم تبعية لملك متـوج‏.‏ 
والملوك 

TXT metadata:
{'source': 'C:\\Users\\DELL\\OneDrive\\Desktop\\rag-assistant-project\\data\\raw\\raw_text.txt'}


In [5]:

# ============================================================
# 5. Remove first 19 PDF pages only
# ============================================================

PDF_START_PAGE = 19

pdf_documents = pdf_documents[PDF_START_PAGE:]

print(f"PDF pages after removing first {PDF_START_PAGE}: {len(pdf_documents)}")

PDF pages after removing first 19: 238


In [6]:
# ============================================================
# 6. Combine documents
# ============================================================

all_documents = pdf_documents + txt_documents

print(f"Total documents before cleaning: {len(all_documents)}")

Total documents before cleaning: 239


# 3.Cleaning

In [7]:
# ============================================================
# 7. Cleaning patterns
# ============================================================

# Invisible characters
INVISIBLE_CHARS = re.compile(
    r"[\u200e\u200f\u061c\ufeff\u200b]"
)

# Tatweel
TATWEEL = re.compile(
    r"\u0640"
)

# Arabic diacritics
DIACRITICS = re.compile(
    r"[\u064B-\u0652\u0670]"
)

# Multiple spaces / tabs
MULTI_SPACE = re.compile(
    r"[ \t]+"
)

# 3+ new lines
MULTI_NEWLINE = re.compile(
    r"\n{3,}"
)

In [8]:
# ============================================================
# 8. Table of Contents detection
# ============================================================

def is_toc_line(line):

    line = line.strip()

    # TOC title
    if "فهرس الموضوعات" in line or line == "فهرس":
        return True

    # Example:
    # التعريف بموسوعة .......... 7
    if re.match(r"^.+?\.+\s*\d+\s*$", line):
        return True

    # Example:
    # 1- ذكر كذا .......... 25
    if re.match(
        r"^\d+\s*[-ـ]\s*.+?\.+\s*\d+\s*$",
        line
    ):
        return True

    return False

In [9]:
# ============================================================
# 9. PDF header/footer detection
# ============================================================

PDF_HEADER_FOOTER = re.compile(
    r"^\s*(مختصر السيرة النبوية\s*\|\s*\d+|\d+\s*\|\s*مختصر السيرة النبوية)\s*$",
    re.MULTILINE
)


In [10]:
# ============================================================
# 10. Clean documents
# ============================================================

cleaned_documents = []

for i, doc in enumerate(all_documents):

    text = doc.page_content

    before_len = len(text)

    # Remove invisible characters
    text = INVISIBLE_CHARS.sub("", text)

    # Remove Tatweel
    text = TATWEEL.sub("", text)

    # Remove Arabic diacritics
    text = DIACRITICS.sub("", text)

    # Remove PDF header/footer
    text = PDF_HEADER_FOOTER.sub("", text)

    # Remove TOC lines
    lines = [
        line
        for line in text.splitlines()
        if not is_toc_line(line)
    ]

    text = "\n".join(lines)

    # Normalize spaces
    text = MULTI_SPACE.sub(" ", text)

    # Normalize new lines
    text = MULTI_NEWLINE.sub("\n\n", text)

    text = text.strip()

    if not text:
        continue

    # Copy metadata
    metadata = dict(doc.metadata)

    cleaned_doc = Document(
        page_content=text,
        metadata=metadata
    )

    cleaned_documents.append(cleaned_doc)

    after_len = len(text)

    print(
        f"Doc {i}: "
        f"before={before_len} | "
        f"after={after_len} | "
        f"removed={before_len - after_len}"
    )

all_documents = cleaned_documents

print("\nDocuments after cleaning:", len(all_documents))

Doc 0: before=1351 | after=1147 | removed=204
Doc 1: before=1393 | after=1155 | removed=238
Doc 2: before=1470 | after=1280 | removed=190
Doc 3: before=1161 | after=1010 | removed=151
Doc 4: before=1375 | after=1161 | removed=214
Doc 5: before=1315 | after=1071 | removed=244
Doc 6: before=1504 | after=1268 | removed=236
Doc 7: before=1482 | after=1228 | removed=254
Doc 8: before=1485 | after=1255 | removed=230
Doc 9: before=1352 | after=1140 | removed=212
Doc 10: before=1278 | after=1043 | removed=235
Doc 11: before=1416 | after=1183 | removed=233
Doc 12: before=1460 | after=1270 | removed=190
Doc 13: before=1469 | after=1229 | removed=240
Doc 14: before=1251 | after=1053 | removed=198
Doc 15: before=1346 | after=1102 | removed=244
Doc 16: before=498 | after=411 | removed=87
Doc 17: before=1141 | after=971 | removed=170
Doc 18: before=1474 | after=1221 | removed=253
Doc 19: before=1168 | after=1016 | removed=152
Doc 20: before=1056 | after=906 | removed=150
Doc 21: before=1298 | after=

In [11]:
# ============================================================
# 11. Inspect documents
# ============================================================

for i, doc in enumerate(all_documents[:5]):

    print("=" * 80)
    print(f"Document #{i}")
    print("Source:", os.path.basename(doc.metadata.get("source", "unknown")))
    print("Page:", doc.metadata.get("page", "N/A"))
    print()
    print(doc.page_content[:1000])

Document #0
Source: alsyra_alnubawia.pdf
Page: 19

[ 
1- ذكر سرد النسب الزكي 
قال أبو محمد عبد الملك بن هشام النحوي: هذا كتاب سيرة رسول الله صلى 
الله عليه وآله وسلم، محمد بن عبد الله بن عبد المطلب، واسم عبد المطلب: شيبة 
بن هاشم، واسم هاشم : عمرو بن عبد م ناف، واسم عبد مناف: الم غيرة بن ق صي، 
واسم قصي: زيد بن ك لاب بن م ر ة بن كعب بن ل ؤي بن غالب بن ف هر بن مالك 
بن النض بن ك نانة بن خ زيمة بن م در ك ة، واسم مدركة : عامر بن إلياس بن م ض 
بن ن زار بن م ع د بن عدنان . 
وأنا إن شاء الله مبتدئ هذا الكتاب بذكر إسماعيل بن إبراهيم، ومن ولد 
رسول الله صلى الله عليه وآله وسلم من ولده، وأولادهم لأصلابهم، الأول 
فالأول، من إسماعيل إلى رسول الله صلى الله عليه وآله وسلم، وما يعرض من 
حديثهم، وتارك ذكر غيرهم من ولد إسماعيل، على هذه الجهة للاختصار، إلى 
حديث سيرة رسول الله ﷺ، وتارك بعض ما ذكره ابن إسحاق في هذ ا الكتاب، 
مما ليس لرسول الله صلى الله عليه وآله وسلم فيه ذكر، ولا نزل فيه من القرآن 
شيء، وليس سببا لشيء من هذا الكتاب، ولا تفسيرا له، ولا شاهدا عليه؛ لما 
ذكرت من الاختصار، وأشعارا ذكرها لم 

# 3.Chunking

In [12]:
# ============================================================
# 12. Text splitting
# ============================================================

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        "؟",
        "!",
        "،",
        " ",
        ""
    ]
)

# ============================================================
# 13. Create chunks + metadata
# ============================================================

chunks = []
chunk_metadatas = []

global_chunk_index = 0

for doc in all_documents:

    pieces = splitter.split_text(
        doc.page_content
    )

    source_name = os.path.basename(
        doc.metadata.get("source", "unknown")
    )

    original_page = doc.metadata.get("page", -1)

    for local_index, piece in enumerate(pieces):

        chunks.append(piece)

        chunk_metadatas.append({
            "source": source_name,

            # Keep LangChain page number internally
            "page": original_page,

            # Unique global chunk id
            "chunk_index": global_chunk_index,

            # Chunk number inside the original document/page
            "local_chunk_index": local_index,
        })

        global_chunk_index += 1


print(f"Total chunks: {len(chunks):,}")

avg_len = sum(len(c) for c in chunks) / len(chunks)

print(f"Average chunk length: {avg_len:.0f} characters")

Total chunks: 1,062
Average chunk length: 918 characters


In [13]:
# Print a few sample chunks with their metadata to sanity-check the output
SAMPLE_SIZE = 5

for i in range(SAMPLE_SIZE):
    print(f"--- Chunk #{i} ---")
    print(f"Source: {chunk_metadatas[i]['source']}  |  Page: {chunk_metadatas[i]['page']}  |  Chunk index: {chunk_metadatas[i]['chunk_index']}")
    print(f"Length: {len(chunks[i])} chars")
    print(chunks[i][:300].replace(chr(10), " "))
    print("-" * 80)

# Also print the last few chunks, to check the end of the documents
print("\nLast chunks:")
for i in range(len(chunks) - SAMPLE_SIZE, len(chunks)):
    print(f"--- Chunk #{i} ---")
    print(f"Source: {chunk_metadatas[i]['source']}  |  Page: {chunk_metadatas[i]['page']}  |  Chunk index: {chunk_metadatas[i]['chunk_index']}")
    print(chunks[i][:300].replace(chr(10), " "))
    print("-" * 80)

--- Chunk #0 ---
Source: alsyra_alnubawia.pdf  |  Page: 19  |  Chunk index: 0
Length: 1147 chars
[  1- ذكر سرد النسب الزكي  قال أبو محمد عبد الملك بن هشام النحوي: هذا كتاب سيرة رسول الله صلى  الله عليه وآله وسلم، محمد بن عبد الله بن عبد المطلب، واسم عبد المطلب: شيبة  بن هاشم، واسم هاشم : عمرو بن عبد م ناف، واسم عبد مناف: الم غيرة بن ق صي،  واسم قصي: زيد بن ك لاب بن م ر ة بن كعب بن ل ؤي بن غالب 
--------------------------------------------------------------------------------
--- Chunk #1 ---
Source: alsyra_alnubawia.pdf  |  Page: 20  |  Chunk index: 1
Length: 1155 chars
2- ذكر نذر عبد المطلب ذبح ولد ه  كان عبد المطلب بن هاشم -فيما يزع مون والله أعلم- قد نذر حين لق ي من  قريش ما لقي عند حفر زمزم : لئن و ل د له عشر ة نفر، ثم بلغوا معه حتى يمنعوه،  لي نحرن أحد هم لله عند الكعبة .  فلما ت واف بنوه عشرة ، وعر ف أنهم سيمنعونه، جم عهم ثم أخبر هم بنذر ه،  ودعاهم إلى الوفاء
--------------------------------------------------------------------------------
--- Chunk #2 ---
Source: alsyra_alnubawia.

In [14]:
# ============================================================
# 14. Inspect chunks
# ============================================================

SAMPLE_SIZE = 5

for i in range(min(SAMPLE_SIZE, len(chunks))):

    metadata = chunk_metadatas[i]

    print("=" * 80)
    print(f"Chunk #{i}")

    print(
        "Source:",
        metadata["source"]
    )

    print(
        "Page:",
        metadata["page"]
    )

    print(
        "Chunk index:",
        metadata["chunk_index"]
    )

    print(
        "Length:",
        len(chunks[i])
    )

    print()

    print(
        chunks[i][:500].replace("\n", " ")
    )

Chunk #0
Source: alsyra_alnubawia.pdf
Page: 19
Chunk index: 0
Length: 1147

[  1- ذكر سرد النسب الزكي  قال أبو محمد عبد الملك بن هشام النحوي: هذا كتاب سيرة رسول الله صلى  الله عليه وآله وسلم، محمد بن عبد الله بن عبد المطلب، واسم عبد المطلب: شيبة  بن هاشم، واسم هاشم : عمرو بن عبد م ناف، واسم عبد مناف: الم غيرة بن ق صي،  واسم قصي: زيد بن ك لاب بن م ر ة بن كعب بن ل ؤي بن غالب بن ف هر بن مالك  بن النض بن ك نانة بن خ زيمة بن م در ك ة، واسم مدركة : عامر بن إلياس بن م ض  بن ن زار بن م ع د بن عدنان .  وأنا إن شاء الله مبتدئ هذا الكتاب بذكر إسماعيل بن إبراهيم، ومن ولد  رسول الله
Chunk #1
Source: alsyra_alnubawia.pdf
Page: 20
Chunk index: 1
Length: 1155

2- ذكر نذر عبد المطلب ذبح ولد ه  كان عبد المطلب بن هاشم -فيما يزع مون والله أعلم- قد نذر حين لق ي من  قريش ما لقي عند حفر زمزم : لئن و ل د له عشر ة نفر، ثم بلغوا معه حتى يمنعوه،  لي نحرن أحد هم لله عند الكعبة .  فلما ت واف بنوه عشرة ، وعر ف أنهم سيمنعونه، جم عهم ثم أخبر هم بنذر ه،  ودعاهم إلى الوفاء لله بذلك، فأطاعوه وقالوا: كيف نصنع ؟ قال: ليأخ

# Embeddings 

In [15]:
# ============================================================
# 15. E5 Embedding model
# ============================================================

from langchain_huggingface import HuggingFaceEmbeddings


class E5Embeddings(HuggingFaceEmbeddings):

    def embed_documents(self, texts):

        texts = [
            f"passage: {text}"
            for text in texts
        ]

        return super().embed_documents(texts)

    def embed_query(self, text):

        text = f"query: {text}"

        return super().embed_query(text)

In [16]:
# ============================================================
# 16. Initialize E5
# ============================================================

embedding_model = E5Embeddings(
    model_name="intfloat/multilingual-e5-base",

    model_kwargs={
        "device": "cpu"
    },

    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 743.47it/s]


Embedding model loaded.


In [17]:
# ============================================================
# 17. Create LangChain Documents
# ============================================================

all_docs = []

for i, (chunk, metadata) in enumerate(
    zip(chunks, chunk_metadatas)
):

    metadata = dict(metadata)

    # Make absolutely sure chunk id is unique
    metadata["chunk_index"] = i

    all_docs.append(
        Document(
            page_content=chunk,
            metadata=metadata
        )
    )

print(f"LangChain documents: {len(all_docs):,}")

LangChain documents: 1,062


In [18]:
# ============================================================
# 18. Build FAISS vector store
# ============================================================

vector_store = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

print(
    f"FAISS index contains "
    f"{vector_store.index.ntotal:,} vectors"
)

FAISS index contains 1,062 vectors


# 5.Retrieval

In [19]:
# ============================================================
# 19. BM25 Retriever
# ============================================================

bm25_retriever = BM25Retriever.from_documents(
    all_docs
)

bm25_retriever.k = 8

print("BM25 retriever ready.")

BM25 retriever ready.


In [20]:
# ============================================================
# 20. Vector Retriever
# ============================================================

vector_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 8
    }
)

print("Vector retriever ready.")

Vector retriever ready.


In [21]:
# ============================================================
# 21. Hybrid Retriever
# ============================================================

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        vector_retriever
    ],

    weights=[
        0.45,   # BM25
        0.55    # Vector / E5
    ]
)

print("Hybrid retriever ready.")

Hybrid retriever ready.


In [22]:
# ============================================================
# 22. Retrieval function
# ============================================================

def retrieve(question: str, k: int = 6):

    # Retrieve more candidates internally
    internal_k = max(k, 8)

    bm25_retriever.k = internal_k

    vector_retriever.search_kwargs["k"] = internal_k

    results = hybrid_retriever.invoke(
        question
    )

    return results[:k]

In [23]:
# ============================================================
# 23. Source helpers
# ============================================================

def get_page_label(doc):

    page = doc.metadata.get("page", -1)

    if page is None or page == -1:
        return None

    # PyPDFLoader uses 0-based page numbers
    return page + 1

In [24]:
def doc_id(doc):

    source = doc.metadata.get(
        "source",
        "unknown"
    )

    page = doc.metadata.get(
        "page",
        -1
    )

    chunk_index = doc.metadata.get(
        "chunk_index",
        "unknown"
    )

    return (
        f"{source}"
        f"::page={page}"
        f"::chunk={chunk_index}"
    )

In [25]:
def format_source(doc):

    source = doc.metadata.get(
        "source",
        "unknown"
    )

    page_label = get_page_label(doc)

    chunk_index = doc.metadata.get(
        "chunk_index",
        "unknown"
    )

    if page_label is not None:

        return (
            f"{source} "
            f"(صفحة {page_label}, "
            f"chunk {chunk_index})"
        )

    return (
        f"{source} "
        f"(chunk {chunk_index})"
    )

In [26]:
def format_sources(results):

    sources = []

    seen = set()

    for doc in results:

        source = format_source(doc)

        if source not in seen:

            seen.add(source)
            sources.append(source)

    return sources

# Test Retrieval

In [27]:
# ============================================================
# 24. Test questions
# ============================================================

test_questions = [

    "متى وُلد النبي صلى الله عليه وسلم؟",

    "من هي أم النبي صلى الله عليه وسلم؟",

    "ماذا حدث في غزوة بدر؟",

    "من هم ملوك اليمن قبل الإسلام؟",

    "ما هي هجرة الحبشة الأولى؟",

    "من هو أبو بكر الصديق؟",

    "ماذا حدث في صلح الحديبية؟",

    "كيف كانت الحالة السياسية في جزيرة العرب قبل الإسلام؟",

    "ما هي غزوة الأحزاب؟",

    "متى فُتحت مكة؟",

    "من هي أزواج النبي صلى الله عليه وسلم؟",

    "ما هي غزوة أحد؟",

    "من هو أبو سفيان بن حرب؟",

    "كيف كان أول نزول للوحي على النبي صلى الله عليه وسلم؟",

    "ما هي قصة الإسراء والمعراج؟",

    "متى كانت غزوة خيبر؟",

    "من هو النجاشي ملك الحبشة؟",

    "ما هي حادثة الإفك؟",

    "متى توفي النبي صلى الله عليه وسلم؟",

    "ما هي غزوة تبوك؟",
]

In [28]:
# ============================================================
# 25. Basic retrieval test
# ============================================================

for q in test_questions:

    results = retrieve(q, k=6)

    print("=" * 100)

    print("السؤال:")
    print(q)

    print("\nالمصادر:")

    for source in format_sources(results):
        print(" -", source)

    print("\nأفضل قطعة:")

    if results:

        print(
            results[0].page_content[:500]
            .replace("\n", " ")
        )

    else:

        print("No results.")

السؤال:
متى وُلد النبي صلى الله عليه وسلم؟

المصادر:
 - raw_text.txt (chunk 332)
 - raw_text.txt (chunk 715)
 - raw_text.txt (chunk 348)
 - raw_text.txt (chunk 350)
 - raw_text.txt (chunk 530)
 - raw_text.txt (chunk 1037)

أفضل قطعة:
المولد وأربعون عاما قبل النبوة  المولد  ولد سيد المرسلين صلى الله عليه وسلم بشعب بني هاشم بمكة في صبيحة يوم الاثنين التاسع من شهر ربيع الأول، لأول عام من حادثة الفيل، ولأربعين سنة خلت من ملك كسرى أنوشروان، ويوافق ذلك عشرين أو اثنين وعشرين من شهر أبريل سنة 571 م حسبما حققه العالم الكبير محمد سليمان المنصورفورى رحمه الله .  وروى ابن سعد أن أم رسول الله صلى الله عليه وسلم قالت : لما ولدته خرج من فرجى نور أضاءت له قصور الشام. وروى أحمد والدارمى وغيرهما قريبا من ذلك.  وقد روى أن إرهاصات بالبعثة وقعت 
السؤال:
من هي أم النبي صلى الله عليه وسلم؟

المصادر:
 - raw_text.txt (chunk 1037)
 - raw_text.txt (chunk 1042)
 - raw_text.txt (chunk 1041)
 - raw_text.txt (chunk 1030)
 - raw_text.txt (chunk 1038)
 - alsyra_alnubawia.pdf (صفحة 107, chunk 105)

أفضل قطعة:
3 عائشة ب

# 6.Evaluation

In [29]:
# ============================================================
# 26. Detailed retrieval evaluation
# ============================================================

for q in test_questions:

    results = retrieve(q, k=6)

    print("\n")
    print("=" * 100)
    print("السؤال:", q)
    print("=" * 100)

    for rank, doc in enumerate(
        results,
        start=1
    ):

        print(
            f"\n[{rank}] "
            f"{format_source(doc)}"
        )

        print(
            "-" * 80
        )

        print(
            doc.page_content[:800]
            .replace("\n", " ")
        )

        print()



السؤال: متى وُلد النبي صلى الله عليه وسلم؟

[1] raw_text.txt (chunk 332)
--------------------------------------------------------------------------------
المولد وأربعون عاما قبل النبوة  المولد  ولد سيد المرسلين صلى الله عليه وسلم بشعب بني هاشم بمكة في صبيحة يوم الاثنين التاسع من شهر ربيع الأول، لأول عام من حادثة الفيل، ولأربعين سنة خلت من ملك كسرى أنوشروان، ويوافق ذلك عشرين أو اثنين وعشرين من شهر أبريل سنة 571 م حسبما حققه العالم الكبير محمد سليمان المنصورفورى رحمه الله .  وروى ابن سعد أن أم رسول الله صلى الله عليه وسلم قالت : لما ولدته خرج من فرجى نور أضاءت له قصور الشام. وروى أحمد والدارمى وغيرهما قريبا من ذلك.  وقد روى أن إرهاصات بالبعثة وقعت عند الميلاد، فسقطت أربع عشرة شرفة من إيوان كسرى، وخمدت النار التي يعبدها المجوس، وانهدمت الكنائس حول بحيرة ساوة بعد أن غاضت، روى ذلك الطبرى والبيهقى وغيرهما. وليس له إسناد ثابت، ولم يشهد له تاريخ تلك الأمم مع قوة دواعى التسجيل.  ولما ولدته أمه أرسلت إلى جده عبد المطلب تبشره بحفيده،فجاء مستبشرا ودخل 


[2] raw_text.txt (chunk 715)
------------



### أهم الملاحظات

| السؤال                    | التقييم    | المشكلة                                                                              |
| ------------------------- | ---------- | ------------------------------------------------------------------------------------ |
| متى وُلد النبي ﷺ؟         | جيد جدًا   | أفضل قطعة صحيحة ومباشرة، لكن باقي الـ chunks أغلبها irrelevant                       |
| من هي أم النبي ﷺ؟         | ضعيف       | رجّع معلومات عن زوجات النبي، خصوصًا عائشة، بدل أم النبي وهي آمنة بنت وهب             |
| ماذا حدث في غزوة بدر؟     | متوسط      | رجّع بداية/سياق بدر، لكن الإجابة غير مكتملة ومش مباشرة                               |
| من هم ملوك اليمن؟         | جيد        | الـ chunks مناسبة، لكن السؤال يطلب أسماء/مراحل وقد تحتاج الإجابة تجميع أكثر من chunk |
| ما هي هجرة الحبشة الأولى؟ | جيد نسبيًا | استرجع أجزاء صحيحة، لكن أفضل قطعة لا تحتوي تفاصيل الهجرة كاملة                       |
| من هو أبو بكر الصديق؟     | ضعيف       | رجّع مواقف متفرقة عنه، لكن لم يقدم تعريفًا مباشرًا                                   |
| صلح الحديبية              | جيد        | السياق مرتبط، لكن الـ best chunk تحليلي أكثر من كونه يشرح أحداث الصلح                |
| أزواج النبي ﷺ             | ضعيف/متوسط | رجّع جزءًا من القائمة فقط، وليس كل الزوجات                                           |
| النجاشي                   | جيد جدًا   | القطعة مرتبطة بالنجاشي، لكنها تبدأ من الرسالة إليه ولا تعطي تعريفًا كاملًا           |
| وفاة النبي ﷺ              | ضعيف       | رجّع جزءًا من آخر يوم، لكنه لا يجيب بشكل مباشر عن التاريخ                            |
| غزوة تبوك                 | جيد        | بداية الغزوة وسببها ظاهرين                                                           |
| حادثة الإفك               | جيد جدًا   | الإجابة مرتبطة بشكل واضح بالحادثة                                                    |




**الخلاصة:** الـ RAG محتاج تحسين في:

1. `chunking`
2. `top_k`
3. similarity threshold
4. reranking
5. metadata filtering




# 6.Export

In [30]:
# ============================================================
# 27. Save FAISS vector store to disk
# ============================================================

VECTOR_STORE_PATH = r"C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\processed\faiss_index"

vector_store.save_local(VECTOR_STORE_PATH)

print(f"FAISS index saved to: {VECTOR_STORE_PATH}")
print(f"Vectors saved: {vector_store.index.ntotal:,}")


# ============================================================
# 28. Save chunks + metadata separately (needed to rebuild BM25Retriever)
# ============================================================
# FAISS only stores vectors + documents internally, but BM25Retriever
# needs the raw text again to rebuild its index on the backend side.
# We save all_docs (page_content + metadata) so the backend can load
# them without redoing the PDF/TXT loading, cleaning, and chunking steps.

import pickle

DOCS_PATH = r"C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\processed\all_docs.pkl"

with open(DOCS_PATH, "wb") as f:
    pickle.dump(all_docs, f)

print(f"Documents (for BM25) saved to: {DOCS_PATH}")
print(f"Total documents saved: {len(all_docs):,}")

FAISS index saved to: C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\processed\faiss_index
Vectors saved: 1,062
Documents (for BM25) saved to: C:\Users\DELL\OneDrive\Desktop\rag-assistant-project\data\processed\all_docs.pkl
Total documents saved: 1,062
